# Figure 6 and Figure S6: spatial stability in pPRH and T-maze

Main panels **6d–g and 6i–n** and supplementary **S6c–f**. Historical S3 stability filenames correspond to current S6, not the excluded manipulation-validation S3.

This notebook follows **Xu et al., manuscript**. Data loading, model fitting, panel selection and plotting are shown below as executable steps. Shared scientific methods remain in `src/`; this notebook does not delegate the figure to `run_figure()`.

Start with the separate manuscript source-data files listed in `data/required_files.csv`. These are analysis source tables and frozen map/embedding arrays, rather than raw movies. The aligned-data computation examples are in `00_recompute_metrics.ipynb`.

Run cells from top to bottom. `REFIT_STATISTICS=True` refits the manuscript models; set it to `False` to use the frozen estimates. Outputs are saved separately from inputs.

## 1. Paths and execution parameters

Set `PPRH_DATA_DIR` or edit `DATA_ROOT`. The default refits statistical models. No raw neural resampling is triggered here.

In [ ]:
import os
import sys
from pathlib import Path

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "main.py").is_file() and (p / "src/paper.py").is_file())
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import SVG, Image, display
from paper_io import read_table, save_panel as _save_panel, write_tables, p_symbol, write_run_record

DATA_ROOT = Path(os.environ.get("PPRH_DATA_DIR", str(ROOT / "data/input")))
OUTPUT_ROOT = Path(os.environ.get("PPRH_OUTPUT_DIR", str(ROOT / "outputs/notebooks")))
REFIT_STATISTICS = os.environ.get("PPRH_REFIT_STATISTICS", "1") == "1"
FORMATS = ("svg", "png")
DPI = 300

data_root = DATA_ROOT
output = OUTPUT_ROOT / "Fig6"
refit, formats, dpi = REFIT_STATISTICS, FORMATS, DPI
output.mkdir(parents=True, exist_ok=True)
if not data_root.is_dir():
    raise FileNotFoundError("Supply the separate manuscript inputs through DATA_ROOT; see data/README.md.")


## 2. Analysis contract

C/P comparisons retain recording day or day-pair and lateral-deviation adjustment. Single-neuron/PV correlations are displayed as raw `r` and modeled using Fisher `z`. The main 6j panel uses the accepted **adjacent-day** PV heatmap replacement. T-maze C/I comparisons in S6 retain their own task, route and day-pair labels.

## 3. Notebook functions

The export function saves and previews each panel. Where needed, the source-table loader or annotation helper is defined here in full. Numerical estimators and shared plotting functions are imported in the relevant analysis cells.

In [ ]:
def save_panel(fig, output_root, panel, description, **kwargs):
    """Save one panel with the shared export method, then preview its result."""
    paths = _save_panel(fig, output_root, panel, description, **kwargs)
    svg = next((p for p in paths if p.endswith(".svg")), None)
    png = next((p for p in paths if p.endswith(".png")), None)
    if svg:
        display(SVG(filename=svg))
    elif png:
        display(Image(filename=png))
    return paths


In [ ]:
def _stability_cp_sources(data_root):

    def read(name):
        return read_table(data_root, 'fig6/source_data/' + name + '.csv')
    return {'fig6_within': {'pv_bin': read('Fig6_within_pv_bin'), 'pv_mouse_day': read('Fig6_within_pv_mouse_day'), 'single_neuron': read('Fig6_within_single_neuron')}, 'fig6_interday': {'pv_bin': read('Fig6_interday_pv_bin'), 'pv_mouse_pair': read('Fig6_interday_pv_mouse_pair'), 'single_neuron': read('Fig6_interday_single_neuron')}, 'figs3': {'pv_bin': read('FigS3_interday_pv_bin'), 'pv_mouse_pair': read('FigS3_interday_pv_mouse_pair'), 'single_neuron': read('FigS3_interday_single_neuron')}}


In [ ]:
def _activity_pair(data_root, stem):
    folder = Path(data_root) / 'fig6/source_data/heatmap_matrices'
    return {name: np.load(folder / f'{stem}_{name}.npy') for name in ('map_a_display', 'map_b_display')}


## 4. Source data, statistics and panel-by-panel plots

Each panel cell names its source, contrast and display parameters. Use the current panel names below; historical filenames remain unchanged for traceability. Model fitting can take longer than figure rendering.

### Shared methods used below

In [ ]:
from figure_helpers import figure6 as h


### Read the source data

In [ ]:
from figure_helpers import figure6_adjacent as adj

sources = _stability_cp_sources(data_root)

statistics = read_table(data_root, 'fig6/stats/Fig6_FigS3_primary_adjusted_estimands_raw_P.csv')


In [ ]:
display(statistics.head())


### Fit the manuscript models or retain the frozen estimates

In [ ]:
if refit:
    contrasts, coefficients, _ = h.fit_models(sources, maxiter=1000)
    statistics = contrasts[contrasts.model_role.eq('adjusted')]
    write_tables({'stability_cp_refitted_estimands': contrasts, 'stability_cp_refitted_coefficients': coefficients}, Path(output) / 'statistics')


In [ ]:
display(statistics.head())


### Fig6 d — prepare panel inputs

In [ ]:
exports = []

payload = {day: _activity_pair(data_root, f'Fig6C_P_CL_D{day}_odd_even') for day in (1, 3, 5)}


### Fig6 d — draw and export

In [ ]:
fig = h.plot_within_activity_grid(payload)

exports += save_panel(fig, output, 'Fig6_d', 'odd_even_activity_maps', old_label='C', formats=formats, dpi=dpi)


### Fig6 i — prepare panel inputs

In [ ]:
payload = {f'D{a}-D{b}': _activity_pair(data_root, f'Fig6G_P_CL_D{a}-D{b}') for a, b in ((1, 2), (4, 5))}


### Fig6 i — draw and export

In [ ]:
fig = h.plot_interday_activity_grid(payload, groups=('P',), panel_letter='I')

exports += save_panel(fig, output, 'Fig6_i', 'adjacent_day_activity_maps', formats=formats, dpi=dpi)


### Fig6 e — draw and export

In [ ]:
fig = h.plot_pv_heatmap(sources['fig6_within']['pv_bin'], groups=('P',), condition='CL', mode='within', panel_letter='E', vmin=0.35, vmax=0.85)

exports += save_panel(fig, output, 'Fig6_e', 'within_PV_heatmap', formats=formats, dpi=dpi)


### Fig6 j — prepare panel inputs

In [ ]:
adjacent_rows = read_table(data_root, 'fig6j/source_data/Fig6J_adjacent_interday_locked_mouse_bin_rows.csv')


### Fig6 j — draw and export

In [ ]:
fig = adj.plot_adjacent_fig6j(adjacent_rows)

exports += save_panel(fig, output, 'Fig6_j', 'adjacent_PV_heatmap', formats=formats, dpi=dpi)


### Fig6 f — prepare panel inputs

In [ ]:
panel, analysis, metric = ('f', 'within', 'single_neuron')

key = 'fig6_within' if analysis == 'within' else 'fig6_interday'

obs = sources[key]['single_neuron' if metric == 'single_neuron' else 'pv_mouse_day' if analysis == 'within' else 'pv_mouse_pair']

if analysis != 'within':
    obs = obs[obs.pair_mode.eq(analysis)]


### Fig6 f — draw and export

In [ ]:
fig = h.plot_condition_curve_panel(obs, statistics, figure='Fig. 6', analysis=analysis, metric=metric, time_col='day' if analysis == 'within' else 'time_index', x_values=(1, 2, 3, 4, 5) if analysis == 'within' else (1, 2, 3, 4), x_labels=('1', '2', '3', '4', '5') if analysis == 'within' else ('D1-D2', 'D2-D3', 'D3-D4', 'D4-D5') if analysis == 'adjacent' else ('D1-D2', 'D1-D3', 'D1-D4', 'D1-D5'), groups=('C', 'P'), group_labels=h.FIG6_LABELS, conditions=('CL', 'CR'), condition_labels=('Left', 'Right'), ylabel='Single-neuron spatial activity correlation (r)' if metric == 'single_neuron' else 'PV correlation (r)', xlabel='Day' if analysis == 'within' else 'Day pair', panel_letter=panel.upper(), individual_scatter=metric == 'pv')

exports += save_panel(fig, output, f'Fig6_{panel}', 'spatial_stability', formats=formats, dpi=dpi)


### Fig6 g — prepare panel inputs

In [ ]:
panel, analysis, metric = ('g', 'within', 'pv')

key = 'fig6_within' if analysis == 'within' else 'fig6_interday'

obs = sources[key]['single_neuron' if metric == 'single_neuron' else 'pv_mouse_day' if analysis == 'within' else 'pv_mouse_pair']

if analysis != 'within':
    obs = obs[obs.pair_mode.eq(analysis)]


### Fig6 g — draw and export

In [ ]:
fig = h.plot_condition_curve_panel(obs, statistics, figure='Fig. 6', analysis=analysis, metric=metric, time_col='day' if analysis == 'within' else 'time_index', x_values=(1, 2, 3, 4, 5) if analysis == 'within' else (1, 2, 3, 4), x_labels=('1', '2', '3', '4', '5') if analysis == 'within' else ('D1-D2', 'D2-D3', 'D3-D4', 'D4-D5') if analysis == 'adjacent' else ('D1-D2', 'D1-D3', 'D1-D4', 'D1-D5'), groups=('C', 'P'), group_labels=h.FIG6_LABELS, conditions=('CL', 'CR'), condition_labels=('Left', 'Right'), ylabel='Single-neuron spatial activity correlation (r)' if metric == 'single_neuron' else 'PV correlation (r)', xlabel='Day' if analysis == 'within' else 'Day pair', panel_letter=panel.upper(), individual_scatter=metric == 'pv')

exports += save_panel(fig, output, f'Fig6_{panel}', 'spatial_stability', formats=formats, dpi=dpi)


### Fig6 k — prepare panel inputs

In [ ]:
panel, analysis, metric = ('k', 'adjacent', 'single_neuron')

key = 'fig6_within' if analysis == 'within' else 'fig6_interday'

obs = sources[key]['single_neuron' if metric == 'single_neuron' else 'pv_mouse_day' if analysis == 'within' else 'pv_mouse_pair']

if analysis != 'within':
    obs = obs[obs.pair_mode.eq(analysis)]


### Fig6 k — draw and export

In [ ]:
fig = h.plot_condition_curve_panel(obs, statistics, figure='Fig. 6', analysis=analysis, metric=metric, time_col='day' if analysis == 'within' else 'time_index', x_values=(1, 2, 3, 4, 5) if analysis == 'within' else (1, 2, 3, 4), x_labels=('1', '2', '3', '4', '5') if analysis == 'within' else ('D1-D2', 'D2-D3', 'D3-D4', 'D4-D5') if analysis == 'adjacent' else ('D1-D2', 'D1-D3', 'D1-D4', 'D1-D5'), groups=('C', 'P'), group_labels=h.FIG6_LABELS, conditions=('CL', 'CR'), condition_labels=('Left', 'Right'), ylabel='Single-neuron spatial activity correlation (r)' if metric == 'single_neuron' else 'PV correlation (r)', xlabel='Day' if analysis == 'within' else 'Day pair', panel_letter=panel.upper(), individual_scatter=metric == 'pv')

exports += save_panel(fig, output, f'Fig6_{panel}', 'spatial_stability', formats=formats, dpi=dpi)


### Fig6 l — prepare panel inputs

In [ ]:
panel, analysis, metric = ('l', 'adjacent', 'pv')

key = 'fig6_within' if analysis == 'within' else 'fig6_interday'

obs = sources[key]['single_neuron' if metric == 'single_neuron' else 'pv_mouse_day' if analysis == 'within' else 'pv_mouse_pair']

if analysis != 'within':
    obs = obs[obs.pair_mode.eq(analysis)]


### Fig6 l — draw and export

In [ ]:
fig = h.plot_condition_curve_panel(obs, statistics, figure='Fig. 6', analysis=analysis, metric=metric, time_col='day' if analysis == 'within' else 'time_index', x_values=(1, 2, 3, 4, 5) if analysis == 'within' else (1, 2, 3, 4), x_labels=('1', '2', '3', '4', '5') if analysis == 'within' else ('D1-D2', 'D2-D3', 'D3-D4', 'D4-D5') if analysis == 'adjacent' else ('D1-D2', 'D1-D3', 'D1-D4', 'D1-D5'), groups=('C', 'P'), group_labels=h.FIG6_LABELS, conditions=('CL', 'CR'), condition_labels=('Left', 'Right'), ylabel='Single-neuron spatial activity correlation (r)' if metric == 'single_neuron' else 'PV correlation (r)', xlabel='Day' if analysis == 'within' else 'Day pair', panel_letter=panel.upper(), individual_scatter=metric == 'pv')

exports += save_panel(fig, output, f'Fig6_{panel}', 'spatial_stability', formats=formats, dpi=dpi)


### Fig6 m — prepare panel inputs

In [ ]:
panel, analysis, metric = ('m', 'anchor', 'single_neuron')

key = 'fig6_within' if analysis == 'within' else 'fig6_interday'

obs = sources[key]['single_neuron' if metric == 'single_neuron' else 'pv_mouse_day' if analysis == 'within' else 'pv_mouse_pair']

if analysis != 'within':
    obs = obs[obs.pair_mode.eq(analysis)]


### Fig6 m — draw and export

In [ ]:
fig = h.plot_condition_curve_panel(obs, statistics, figure='Fig. 6', analysis=analysis, metric=metric, time_col='day' if analysis == 'within' else 'time_index', x_values=(1, 2, 3, 4, 5) if analysis == 'within' else (1, 2, 3, 4), x_labels=('1', '2', '3', '4', '5') if analysis == 'within' else ('D1-D2', 'D2-D3', 'D3-D4', 'D4-D5') if analysis == 'adjacent' else ('D1-D2', 'D1-D3', 'D1-D4', 'D1-D5'), groups=('C', 'P'), group_labels=h.FIG6_LABELS, conditions=('CL', 'CR'), condition_labels=('Left', 'Right'), ylabel='Single-neuron spatial activity correlation (r)' if metric == 'single_neuron' else 'PV correlation (r)', xlabel='Day' if analysis == 'within' else 'Day pair', panel_letter=panel.upper(), individual_scatter=metric == 'pv')

exports += save_panel(fig, output, f'Fig6_{panel}', 'spatial_stability', formats=formats, dpi=dpi)


### Fig6 n — prepare panel inputs

In [ ]:
panel, analysis, metric = ('n', 'anchor', 'pv')

key = 'fig6_within' if analysis == 'within' else 'fig6_interday'

obs = sources[key]['single_neuron' if metric == 'single_neuron' else 'pv_mouse_day' if analysis == 'within' else 'pv_mouse_pair']

if analysis != 'within':
    obs = obs[obs.pair_mode.eq(analysis)]


### Fig6 n — draw and export

In [ ]:
fig = h.plot_condition_curve_panel(obs, statistics, figure='Fig. 6', analysis=analysis, metric=metric, time_col='day' if analysis == 'within' else 'time_index', x_values=(1, 2, 3, 4, 5) if analysis == 'within' else (1, 2, 3, 4), x_labels=('1', '2', '3', '4', '5') if analysis == 'within' else ('D1-D2', 'D2-D3', 'D3-D4', 'D4-D5') if analysis == 'adjacent' else ('D1-D2', 'D1-D3', 'D1-D4', 'D1-D5'), groups=('C', 'P'), group_labels=h.FIG6_LABELS, conditions=('CL', 'CR'), condition_labels=('Left', 'Right'), ylabel='Single-neuron spatial activity correlation (r)' if metric == 'single_neuron' else 'PV correlation (r)', xlabel='Day' if analysis == 'within' else 'Day pair', panel_letter=panel.upper(), individual_scatter=metric == 'pv')

exports += save_panel(fig, output, f'Fig6_{panel}', 'spatial_stability', formats=formats, dpi=dpi)


### FigS6 c — prepare panel inputs

In [ ]:
payload = {g: {f'D{a}-D{b}': _activity_pair(data_root, f'FigS3B_{g}_CLC_D{a}-D{b}') for a, b in ((1, 2), (2, 3), (3, 4), (4, 5))} for g in ('C', 'I')}


### FigS6 c — draw and export

In [ ]:
fig = h.plot_interday_activity_grid(payload, groups=('C', 'I'), panel_letter='C')

exports += save_panel(fig, output, 'FigS6_c', 'adjacent_activity_maps', formats=formats, dpi=dpi)


### FigS6 d — prepare panel inputs

In [ ]:
obs = sources['figs3']['pv_bin']


### FigS6 d — draw and export

In [ ]:
fig = h.plot_pv_heatmap(obs[obs.pair_mode.eq('adjacent')], groups=('C', 'I'), condition='left', mode='adjacent', panel_letter='D', vmin=0.05, vmax=0.6)

exports += save_panel(fig, output, 'FigS6_d', 'adjacent_PV_heatmap', formats=formats, dpi=dpi)


### FigS6 e — prepare panel inputs

In [ ]:
panel, mode = ('e', 'adjacent')


### FigS6 e — draw and export

In [ ]:
fig = h.plot_figs3_metric_pair(sources['figs3'], statistics, mode=mode, panel_letter=panel.upper())

exports += save_panel(fig, output, f'FigS6_{panel}', 'spatial_stability', formats=formats, dpi=dpi)


### FigS6 f — prepare panel inputs

In [ ]:
panel, mode = ('f', 'anchor')


### FigS6 f — draw and export

In [ ]:
fig = h.plot_figs3_metric_pair(sources['figs3'], statistics, mode=mode, panel_letter=panel.upper())

exports += save_panel(fig, output, f'FigS6_{panel}', 'spatial_stability', formats=formats, dpi=dpi)


## 5. Export record

Panel files and any refitted model tables are saved under the figure output directory. The record identifies the selected manuscript workflow and whether statistics were refitted.

In [ ]:
write_run_record(output / "run.json", figure="6", refit=refit, exports=exports)
display(pd.DataFrame({"exported_file": [str(Path(p).relative_to(output)) for p in exports]}))
